In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
df = pd.read_parquet('../data/raw/cnes_leitos_PR_2023_01.parquet')
print (f"Dados carregados: {len(df)} linhas")

Dados carregados: 3267 linhas


In [12]:
df.head(5)

,CNES,CODUFMUN,REGSAUDE,MICR_REG,DISTRSAN,DISTRADM,TPGESTAO,PF_PJ,CPF_CNPJ,NIV_DEP,...,NIV_HIER,TERCEIRO,TP_LEITO,CODLEITO,QT_EXIST,QT_CONTR,QT_SUS,QT_NSUS,COMPETEN,NAT_JUR
0,0013102,410020,2,410020,,,D,3,00000000000000,3,...,,,2,33,3,0,3,0,202301,1244
1,0013129,410030,02,,,,D,3,00000000000000,3,...,,,5,45,1,0,1,0,202301,1244
2,0013129,410030,02,,,,D,3,00000000000000,3,...,,,4,43,1,0,1,0,202301,1244
3,0013129,410030,02,,,,D,3,00000000000000,3,...,,,2,33,10,0,10,0,202301,1244
4,2733528,410045,011,,,,M,3,00000000000000,3,...,,,4,10,2,0,2,0,202301,1244


In [13]:
df.shape


(3267, 28)

In [14]:
df.columns.to_list()

['CNES',
 'CODUFMUN',
 'REGSAUDE',
 'MICR_REG',
 'DISTRSAN',
 'DISTRADM',
 'TPGESTAO',
 'PF_PJ',
 'CPF_CNPJ',
 'NIV_DEP',
 'CNPJ_MAN',
 'ESFERA_A',
 'ATIVIDAD',
 'RETENCAO',
 'NATUREZA',
 'CLIENTEL',
 'TP_UNID',
 'TURNO_AT',
 'NIV_HIER',
 'TERCEIRO',
 'TP_LEITO',
 'CODLEITO',
 'QT_EXIST',
 'QT_CONTR',
 'QT_SUS',
 'QT_NSUS',
 'COMPETEN',
 'NAT_JUR']

In [15]:
df.dtypes

,0
CNES,object
CODUFMUN,object
REGSAUDE,object
MICR_REG,object
DISTRSAN,object
DISTRADM,object
TPGESTAO,object
PF_PJ,object
CPF_CNPJ,object
NIV_DEP,object


In [16]:
df.nunique()

,0
CNES,518
CODUFMUN,238
REGSAUDE,45
MICR_REG,11
DISTRSAN,11
DISTRADM,3
TPGESTAO,3
PF_PJ,1
CPF_CNPJ,355
NIV_DEP,2


In [17]:
df.describe(include='all')

,CNES,CODUFMUN,REGSAUDE,MICR_REG,DISTRSAN,DISTRADM,TPGESTAO,PF_PJ,CPF_CNPJ,NIV_DEP,...,NIV_HIER,TERCEIRO,TP_LEITO,CODLEITO,QT_EXIST,QT_CONTR,QT_SUS,QT_NSUS,COMPETEN,NAT_JUR
count,3267,3267,3267,3267,3267,3267,3267,3267,3267,3267,...,3267,3267,3267,3267,3267,3267,3267,3267,3267,3267
unique,518,238,45,11,11,3,3,1,355,2,...,1,1,7,64,88,1,77,52,1,21
top,7758391,410690,02,,,,M,3,00000000000000,1,...,,,1,33,2,0,0,0,202301,3999
freq,42,567,715,3123,2690,3198,1241,3267,658,2485,...,3267,3267,1090,376,537,3267,1032,1410,3267,1177


In [18]:
print(f"\n📌 5.2 TIPOS DE LEITOS:")
print(df['TP_LEITO'].value_counts())


📌 5.2 TIPOS DE LEITOS:
TP_LEITO
1    1090
2     784
3     414
4     410
5     374
6     110
7      85
Name: count, dtype: int64


In [19]:
df.columns = [col.upper() for col in df.columns]
print(f"Colunas padronizadas para maiúsculas")

Colunas padronizadas para maiúsculas


In [20]:
colunas_uteis = [
    'CNES',          # ID do hospital (chave)
    'CODUFMUN',      # Código do município
    'TP_LEITO',      # Tipo de leito
    'QT_EXIST',      # Total de leitos
    'QT_CONTR',      # Leitos contratados
    'QT_SUS',        # Leitos SUS
    'QT_NSUS',       # Leitos não SUS
    'ESFERA_A',      # Esfera (público, privado)
    'TP_UNID',       # Tipo de unidade
    'NIV_HIER',      # Nível hierárquico
    'NAT_JUR',       # Natureza jurídica
]

colunas_presentes = [col for col in colunas_uteis if col in df.columns]
colunas_faltando = [col for col in colunas_uteis if col not in df.columns]

print(f"Colunas presentes: {len(colunas_presentes)}")
print(f"Colunas faltando: {len(colunas_faltando)}")

if colunas_faltando:
    print(f"Colunas não encontradas: {colunas_faltando[:5]}")


Colunas presentes: 11
Colunas faltando: 0


In [21]:
df_selecionado = df[colunas_presentes].copy()

In [22]:
colunas_leitos = ['QT_EXIST', 'QT_CONTR', 'QT_SUS', 'QT_NSUS']
for col in colunas_leitos:
    if col in df_selecionado.columns:
        df_selecionado[col] = pd.to_numeric(df_selecionado[col], errors='coerce').fillna(0).astype(int)
        print(f"   • {col}: convertido para int")

   • QT_EXIST: convertido para int
   • QT_CONTR: convertido para int
   • QT_SUS: convertido para int
   • QT_NSUS: convertido para int


In [23]:
colunas_agrupar = [col for col in df_selecionado.columns if col not in colunas_leitos]
colunas_somar = [col for col in df_selecionado.columns if col in colunas_leitos]

print(f"   • Colunas para agrupar: {colunas_agrupar}")
print(f"   • Colunas para somar: {colunas_somar}")

# Agrupar
df_agrupado = df_selecionado.groupby(colunas_agrupar)[colunas_somar].sum().reset_index()

# Padronizar ID (7 dígitos)
df_agrupado['CNES'] = df_agrupado['CNES'].astype(str).str.zfill(7)

print(f"   • Hospitais únicos: {len(df_agrupado):,}")
print(f"   • Colunas: {df_agrupado.columns.tolist()}")

   • Colunas para agrupar: ['CNES', 'CODUFMUN', 'TP_LEITO', 'ESFERA_A', 'TP_UNID', 'NIV_HIER', 'NAT_JUR']
   • Colunas para somar: ['QT_EXIST', 'QT_CONTR', 'QT_SUS', 'QT_NSUS']
   • Hospitais únicos: 1,717
   • Colunas: ['CNES', 'CODUFMUN', 'TP_LEITO', 'ESFERA_A', 'TP_UNID', 'NIV_HIER', 'NAT_JUR', 'QT_EXIST', 'QT_CONTR', 'QT_SUS', 'QT_NSUS']


In [24]:

for col in colunas_somar:
    if col in df_agrupado.columns:
        print(f"   • {col}:")
        print(f"      - Total: {df_agrupado[col].sum():,}")
        print(f"      - Média: {df_agrupado[col].mean():.2f}")
        print(f"      - Máximo: {df_agrupado[col].max():,}")
        print(f"      - Hospitais com 0: {(df_agrupado[col] == 0).sum()}")

   • QT_EXIST:
      - Total: 30,447
      - Média: 17.73
      - Máximo: 410
      - Hospitais com 0: 0
   • QT_CONTR:
      - Total: 0
      - Média: 0.00
      - Máximo: 0
      - Hospitais com 0: 1717
   • QT_SUS:
      - Total: 20,721
      - Média: 12.07
      - Máximo: 400
      - Hospitais com 0: 396
   • QT_NSUS:
      - Total: 9,726
      - Média: 5.66
      - Máximo: 120
      - Hospitais com 0: 833
